# bge-reranker-base - Cross-Encoder Reranking on Amazon SageMaker

Deploys [BAAI/bge-reranker-base](https://huggingface.co/BAAI/bge-reranker-base) from AWS
Marketplace as a SageMaker endpoint inside **your own AWS account**. Your text never
leaves your VPC and there are no external API calls or token limits.

**Model facts** (from the official model card): cross-encoder architecture,
raw logit relevance score per (query, passage) pair, MIT licence.

Unlike bi-encoder embeddings, a cross-encoder sees the query and passage together,
producing a more accurate relevance score at the cost of higher latency. The typical
pattern is: cheap bi-encoder retrieval (top-20 to top-100), then precise cross-encoder
reranking (top-5 to top-10).

**When to choose bge-reranker-base:**
- Second-stage reranking in a two-stage retrieval pipeline at **$0.08/hr**
- Cases where retrieval precision matters more than latency
- Re-ordering ANN/vector search results before presenting to users

## 1. Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.m5.xlarge"  # the recommended real-time instance
ENDPOINT_NAME = "bge-reranker-base"

session = sagemaker.Session()
try:
    role = sagemaker.get_execution_role()
except ValueError:
    # Running outside SageMaker -- specify your role ARN explicitly
    role = "arn:aws:iam::<ACCOUNT-ID>:role/<SAGEMAKER-ROLE-NAME>"
print("Region:", session.boto_region_name)
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Takes roughly 6-9 minutes. The endpoint bills **$0.08/hr** while it exists,
so do not skip section 6.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Rerank passages

The endpoint accepts `application/json` shaped `{"query": "...", "passages": ["...", ...]}`.
It returns `{"scores": [...]}` — one raw logit per passage.

Scores are raw logits (unbounded floats). A higher score means higher predicted
relevance. Do not apply sigmoid for ranking — logits preserve the correct ordering.

In [ ]:
runtime = boto3.client("sagemaker-runtime")
def rerank(query, passages):
    """
    Score each passage against the query.
    Returns a list of (score, passage) tuples sorted by descending relevance score.
    Scores are raw logits -- higher means more relevant.
    """
    try:
    try:
        response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"query": query, "passages": passages}),
        )
        result = json.loads(response["Body"].read())
    except Exception as e:
        print(f"Error invoking endpoint: {e}")
        raise
    if "scores" not in result:
        raise ValueError(f"Unexpected response format: {result}")
    scored = list(zip(result["scores"], passages))
    return sorted(scored, key=lambda x: x[0], reverse=True)

## 4. Simulate a two-stage retrieval pipeline

In a real pipeline, the first stage (dense retrieval or BM25) returns top-20 to top-100
candidate passages. Here we mock 20 passages and rerank them with the cross-encoder.

In [ ]:
# Mock top-20 passages from a first-stage retriever
query = "How do transformer models handle long sequences?"

top20_passages = [
    "Transformer models use self-attention to process all tokens simultaneously, but memory grows quadratically with sequence length.",
    "Long sequence transformers like Longformer and BigBird use sparse attention patterns to reduce memory to O(n).",
    "The original BERT model was limited to 512 tokens due to positional embedding constraints.",
    "Chunking long documents into overlapping windows is a common workaround for sequence length limits.",
    "Sliding window attention allows the model to attend to local context while maintaining linear complexity.",
    "Flash Attention reorders the attention computation to reduce memory bandwidth requirements significantly.",
    "RNNs process sequences token by token, which avoids the quadratic cost but loses parallelism.",
    "Rotary position embeddings (RoPE) improve length generalisation beyond the training context window.",
    "ALiBi biases attention scores by distance, allowing models to extrapolate to longer sequences at inference time.",
    "Mamba and other state-space models offer linear-time sequence processing as an alternative to attention.",
    "Retrieval-augmented generation sidesteps context limits by fetching relevant chunks before generation.",
    "GPT-4 supports a 128k token context window using efficient attention approximations.",
    "Document chunking strategies include fixed-size windows, sentence boundaries, and paragraph splits.",
    "Positional encodings in the original transformer are sinusoidal and not learned.",
    "The attention mechanism computes queries, keys, and values from the input embeddings.",
    "Layer normalisation is applied before the attention and feed-forward sublayers in modern transformers.",
    "BERT uses WordPiece tokenisation to handle out-of-vocabulary words.",
    "The feedforward network in each transformer layer applies two linear transformations with a ReLU in between.",
    "Tokenisation affects how many tokens a given piece of text consumes and therefore its cost.",
    "Mixture-of-experts models activate only a subset of parameters per token, improving efficiency.",
]

print(f"First-stage candidate count: {len(top20_passages)}")

## 5. Rerank and display score-sorted output

The cross-encoder scores every (query, passage) pair individually and returns raw logits.
Higher logit = more relevant to the query.

In [ ]:
ranked = rerank(query, top20_passages)

print(f"Query: '{query}'\n")
print(f"{'Rank':<6} {'Score':>8}  Passage")
print("-" * 90)
for rank, (score, passage) in enumerate(ranked, start=1):
    # Truncate long passages for display
    display = passage if len(passage) <= 80 else passage[:77] + "..."
    print(f"{rank:<6} {score:>8.4f}  {display}")

print("\nNote: scores are raw logits -- compare relative ordering, not absolute magnitude.")

## 6. Clean up

Delete the endpoint when you are done. It bills $0.08/hr for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)